#### 01. Write a query that returns total revenue (Quantity × Price) and total quantity sold, grouped by Country, sorted highest revenue first — excluding cancelled orders.

In [0]:
from pyspark.sql import functions as f
# from pyspark.sql import desc

df = spark.table("retail_sales_dev.silver.sales_cleaned").select("Country","Quantity","Price","Is_Cancelled").filter(f.col("Is_Cancelled") == False)

df_grouped = df.groupBy("Country").agg(
    f.sum(f.col("Quantity") * f.col("Price")).alias("TotalRevenue"),
    f.sum(f.col("Quantity")).alias("TotalQuantity")
).orderBy(f.desc("TotalRevenue"))

display(df_grouped) 
# display(df)

In [0]:
%sql

SELECT COUNTRY,SUM(QUANTITY*PRICE) AS TotalRevenue,
SUM(QUANTITY) AS TotalQuantity
FROM retail_sales_dev.silver.sales_cleaned
WHERE is_cancelled == FALSE -- OR NOT is_cancelled
group by COUNTRY
order by TotalRevenue DESC;


#### 02. find the customer with the 3rd highest total spend overall — not per country, just globally. Write it in SQL, then PySpark.

In [0]:
%sql
SELECT CustomerID FROM(
SELECT customerid,SUM(Quantity * PRICE),dense_rank() over(ORDER BY SUM(Quantity * PRICE) DESC) AS RN FROM retail_sales_dev.silver.sales_cleaned
WHERE NOT is_cancelled AND CustomerID <> 'UNKNOWN'
group by CustomerID)
WHERE RN = 3


In [0]:
from pyspark.sql.window import Window

df = spark.table("retail_sales_dev.silver.sales_cleaned").filter((f.col("Is_Cancelled") == False) & (f.col("CustomerID")!='UNKNOWN')).select("Country","CustomerID","Quantity","Price")

df_grouped = df.groupBy("CustomerID").agg(
    f.sum(f.col("Quantity") * f.col("Price")).alias("TotalSpend"))
   
# w = Window.orderBy(f.col("TotalSpend").desc())
w = Window.partitionBy(f.col("CustomerID")).orderBy(f.col("TotalSpend").desc())
df_rank = df_grouped.withColumn("rank", f.dense_rank().over(w))

df_filtered = df_rank.select(f.col("CustomerID")).filter(f.col("rank")==3)

display(df_filtered)